# Lab 1 — Databricks Fundamentals & DEV Setup

This notebook is an improved and complete version of the Lab 1 notebook.

It starts from zero and includes:
- Creating schema and Unity Catalog volumes
- Creating raw folders for CSV, JSON, and API files
- Loading `fifa_world_cup_2026_player_performance.csv`
- Creating Bronze, Silver, and multiple Gold Delta tables
- Practicing `select`, `filter`, `join`, and `groupBy`
- Loading data from an external API
- Saving API data as Delta tables
- SQL validation queries
- Dashboard-ready SQL queries
- Comments and markdown explanations

Before running, upload the CSV file here:

`/Volumes/dbr_dev/parvinbadalov/raw_files/lab_01/csv/fifa_world_cup_2026_player_performance.csv`


## 1. Create schema and volumes

This section creates your personal schema and the volumes needed for Lab 1 and later labs.

- `raw_files`: stores raw CSV, JSON, and API files
- `checkpoints`: will be useful for Auto Loader and streaming later
- `documents`: will be useful for RAG / AI capstone later


In [0]:
%sql
USE CATALOG dbr_dev;

CREATE SCHEMA IF NOT EXISTS dbr_dev.parvinbadalov
COMMENT 'Parvin Badalov - Databricks Academy';

CREATE VOLUME IF NOT EXISTS dbr_dev.parvinbadalov.raw_files
COMMENT 'Raw file landing zone for academy labs';

CREATE VOLUME IF NOT EXISTS dbr_dev.parvinbadalov.checkpoints
COMMENT 'Checkpoint and schema tracking location for Auto Loader and streaming';

CREATE VOLUME IF NOT EXISTS dbr_dev.parvinbadalov.documents
COMMENT 'Document storage for RAG and AI capstone labs';


## 2. Define variables and create raw folders

Using variables makes the notebook easier to maintain and re-run.


In [0]:
# Catalog and schema used in the academy workspace.
catalog = "dbr_dev"
schema = "parvinbadalov"

# Base folder paths in Unity Catalog volume.
raw_base_path = f"/Volumes/{catalog}/{schema}/raw_files/lab_01"
raw_csv_path = f"{raw_base_path}/csv"
raw_json_path = f"{raw_base_path}/json"
raw_api_path = f"{raw_base_path}/api"

# Input CSV file path.
csv_file_name = "fifa_world_cup_2026_player_performance.csv"
csv_file_path = f"{raw_csv_path}/{csv_file_name}"

# Bronze table name.
bronze_table = f"{catalog}.{schema}.lab01_fifa_players_bronze"






# Create raw folders.
dbutils.fs.mkdirs(raw_base_path)
dbutils.fs.mkdirs(raw_csv_path)
dbutils.fs.mkdirs(raw_json_path)
dbutils.fs.mkdirs(raw_api_path)

print("CSV file should be uploaded to:", csv_file_path)
print("Bronze table:", bronze_table)



display(dbutils.fs.ls(raw_base_path))


## 3. Load FIFA CSV data

This section loads the FIFA CSV file from the Unity Catalog volume into a Spark DataFrame.


In [0]:
# Read the FIFA CSV dataset.
# inferSchema is okay for Lab 1. In production, an explicit schema is usually better.
fifa_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(csv_file_path)
)

# Preview data and schema.
display(fifa_df)
fifa_df.printSchema()

print("Row count:", fifa_df.count())
print("Column count:", len(fifa_df.columns))


## 4. Create Bronze Delta table

Bronze layer keeps raw data and adds ingestion metadata.

Metadata columns:
- `source_file`
- `ingestion_timestamp`
- `load_date`


In [0]:
from pyspark.sql.functions import (
    col,
    current_timestamp,
    current_date,
    round,
    when,
    sum as spark_sum,
    avg,
    count,
    countDistinct,
    max,
    min,
    lit,
    to_date
)

# Bronze table = raw FIFA data + ingestion metadata.
# In Unity Catalog, use _metadata.file_path instead of input_file_name().
fifa_bronze_df = (
    fifa_df
    .withColumn("source_file", col("_metadata.file_path"))
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("load_date", current_date())
)

# Save as Delta table in Unity Catalog.
fifa_bronze_df.write.mode("overwrite").format("delta").saveAsTable(bronze_table)

display(spark.table(bronze_table))

## 5. Practice Spark DataFrame operations

Lab 1 requires basic Spark operations:
- `select`
- `filter`
- `join`
- `groupBy`

The next cells show multiple examples.


### 5.1 Select examples


In [0]:
selected_df = fifa_bronze_df.select(
    "player_id",
    "player_name",
    "age",
    "nationality",
    "team",
    "position",
    "club_name",
    "market_value_eur",
    "match_id",
    "match_date",
    "opponent_team",
    "tournament_stage",
    "minutes_played",
    "goals",
    "assists",
    "shots",
    "shots_on_target",
    "expected_goals_xg",
    "expected_assists_xa",
    "pass_accuracy",
    "player_rating",
    "performance_score",
    "top_speed_kmh",
    "distance_covered_km"
)

display(selected_df)

In [0]:
# SELECT example 2: choose defensive metrics.
defensive_selected_df = fifa_bronze_df.select(
    "player_id",
    "player_name",
    "team",
    "position",
    "minutes_played",
    "tackles",
    "interceptions",
    "clearances",
    "blocks",
    "defensive_actions",
    "defensive_contribution"
)

display(defensive_selected_df)


### 5.2 Filter examples


In [0]:
# FILTER example 1: players who played at least one minute.
played_df = selected_df.filter(col("minutes_played") > 0)

display(played_df)


In [0]:
# FILTER example 2: players with attacking impact, goals or assists.
attacking_impact_df = selected_df.filter(
    (col("goals") > 0) | (col("assists") > 0)
)

display(attacking_impact_df)


In [0]:
# FILTER example 3: high-rated performances.
high_rating_df = selected_df.filter(col("player_rating") >= 8.0)

display(high_rating_df)


In [0]:
# FILTER example 4: goalkeepers only.
goalkeepers_df = fifa_bronze_df.filter(col("position") == "Goalkeeper")

display(goalkeepers_df)


### 5.3 Join examples

The original notebook already had a team-region join. This version keeps that and adds a position-group join, then uses the joined data in Silver.

This is a small manual lookup table created only for Lab 1 join practice.


In [0]:
# Small lookup table: team to region.
team_region_data = [
    ("Argentina", "South America"),
    ("Brazil", "South America"),
    ("France", "Europe"),
    ("Germany", "Europe"),
    ("Spain", "Europe"),
    ("England", "Europe"),
    ("Portugal", "Europe"),
    ("Netherlands", "Europe"),
    ("Morocco", "Africa"),
    ("Senegal", "Africa"),
    ("Japan", "Asia"),
    ("South Korea", "Asia"),
    ("USA", "North America"),
    ("Mexico", "North America"),
    ("Canada", "North America"),
    ("Poland", "Europe"),
    ("Belgium", "Europe"),
    ("Croatia", "Europe"),
    ("Uruguay", "South America"),
    ("Ecuador", "South America")
]

team_region_df = spark.createDataFrame(team_region_data, ["team", "region"])

display(team_region_df)


In [0]:
# Small lookup table: position to position group.
position_group_data = [
    ("Goalkeeper", "Goalkeeper"),
    ("Defender", "Defensive"),
    ("Midfielder", "Midfield"),
    ("Forward", "Attacking")
]

position_group_df = spark.createDataFrame(position_group_data, ["position", "position_group"])

display(position_group_df)


In [0]:
# JOIN example 1: add region to player data.
joined_region_df = played_df.join(team_region_df, on="team", how="left")

display(joined_region_df)


In [0]:
# JOIN example 2: add position group.
joined_df = joined_region_df.join(position_group_df, on="position", how="left")

display(joined_df)


### 5.4 GroupBy examples


In [0]:
# GROUP BY example 1: goals and assists by team.
team_groupby_df = (
    fifa_bronze_df
    .groupBy("team")
    .agg(
        count("*").alias("records"),
        spark_sum("goals").alias("total_goals"),
        spark_sum("assists").alias("total_assists"),
        round(avg("player_rating"), 2).alias("avg_player_rating")
    )
    .orderBy("total_goals", ascending=False)
)

display(team_groupby_df)


In [0]:
# GROUP BY example 2: performance by position.
position_groupby_df = (
    fifa_bronze_df
    .groupBy("position")
    .agg(
        count("*").alias("records"),
        spark_sum("goals").alias("total_goals"),
        spark_sum("assists").alias("total_assists"),
        round(avg("performance_score"), 2).alias("avg_performance_score")
    )
    .orderBy("avg_performance_score", ascending=False)
)

display(position_groupby_df)


In [0]:
# GROUP BY example 3: performance by tournament stage.
stage_groupby_df = (
    fifa_bronze_df
    .groupBy("tournament_stage")
    .agg(
        count("*").alias("records"),
        spark_sum("goals").alias("total_goals"),
        round(avg("player_rating"), 2).alias("avg_player_rating"),
        max("top_speed_kmh").alias("max_top_speed_kmh")
    )
    .orderBy("total_goals", ascending=False)
)

display(stage_groupby_df)


## 6. Create Silver Delta table

Silver layer is cleaned and enriched.

Changes:
- removes duplicate `player_id + match_id`
- fills missing values
- keeps joined columns `region` and `position_group`
- adds calculated columns:
  - `goal_contribution`
  - `shots_accuracy_pct`
  - `xg_overperformance`
  - `minutes_category`
  - `performance_band`


In [0]:
# Silver table name.

silver_table = f"{catalog}.{schema}.lab01_fifa_players_silver"
print("Silver table:", silver_table)

In [0]:
# Silver table = cleaned and enriched player-match data.
fifa_silver_df = (
    joined_df
    .dropDuplicates(["player_id", "match_id"])
    .fillna({
        "goals": 0,
        "assists": 0,
        "shots": 0,
        "shots_on_target": 0,
        "minutes_played": 0,
        "expected_goals_xg": 0.0,
        "expected_assists_xa": 0.0,
        "pass_accuracy": 0.0,
        "player_rating": 0.0,
        "performance_score": 0.0,
        "market_value_eur": 0,
        "region": "Unknown",
        "position_group": "Unknown"
    })
    .withColumn("goal_contribution", col("goals") + col("assists"))
    .withColumn(
        "shots_accuracy_pct",
        when(col("shots") > 0, round((col("shots_on_target") / col("shots")) * 100, 2))
        .otherwise(0.0)
    )
    .withColumn("xg_overperformance", round(col("goals") - col("expected_goals_xg"), 2))
    .withColumn(
        "minutes_category",
        when(col("minutes_played") >= 75, "High minutes")
        .when(col("minutes_played") >= 30, "Medium minutes")
        .otherwise("Low minutes")
    )
    .withColumn(
        "performance_band",
        when(col("player_rating") >= 8.0, "Excellent")
        .when(col("player_rating") >= 7.0, "Good")
        .when(col("player_rating") >= 6.0, "Average")
        .otherwise("Needs improvement")
    )
)

fifa_silver_df.write.mode("overwrite").format("delta").saveAsTable(silver_table)

display(spark.table(silver_table))


## 7. Create multiple Gold Delta tables

Gold tables are dashboard-ready business summaries.

This version creates five Gold tables:
1. Team summary
2. Position summary
3. Top players
4. Tournament stage summary
5. Value/performance summary


In [0]:
# Gold table names.
gold_team_summary_table = f"{catalog}.{schema}.lab01_fifa_team_summary_gold"
gold_position_summary_table = f"{catalog}.{schema}.lab01_fifa_position_summary_gold"
gold_top_players_table = f"{catalog}.{schema}.lab01_fifa_top_players_gold"
gold_stage_summary_table = f"{catalog}.{schema}.lab01_fifa_stage_summary_gold"
gold_value_summary_table = f"{catalog}.{schema}.lab01_fifa_value_summary_gold"


In [0]:
print("Main Gold table:", gold_team_summary_table)

### 7.1 Gold table — Team summary


In [0]:
# Main dashboard Gold table: team-level summary.
team_summary_df = (
    fifa_silver_df
    .groupBy("team", "nationality", "region")
    .agg(
        count("*").alias("player_match_records"),
        countDistinct("player_id").alias("total_players"),
        countDistinct("match_id").alias("total_matches"),
        spark_sum("goals").alias("total_goals"),
        spark_sum("assists").alias("total_assists"),
        spark_sum("goal_contribution").alias("total_goal_contributions"),
        round(avg("player_rating"), 2).alias("avg_player_rating"),
        round(avg("performance_score"), 2).alias("avg_performance_score"),
        round(avg("pass_accuracy"), 2).alias("avg_pass_accuracy"),
        round(avg("shots_accuracy_pct"), 2).alias("avg_shots_accuracy_pct"),
        max("market_value_eur").alias("highest_market_value_eur"),
        round(avg("market_value_eur"), 2).alias("avg_market_value_eur")
    )
    .orderBy("total_goal_contributions", ascending=False)
)

team_summary_df.write.mode("overwrite").format("delta").saveAsTable(gold_team_summary_table)

display(spark.table(gold_team_summary_table))


### 7.2 Gold table — Position summary


In [0]:
# Position-level Gold table.
position_summary_df = (
    fifa_silver_df
    .groupBy("position", "position_group")
    .agg(
        count("*").alias("player_match_records"),
        countDistinct("player_id").alias("total_players"),
        spark_sum("goals").alias("total_goals"),
        spark_sum("assists").alias("total_assists"),
        spark_sum("goal_contribution").alias("total_goal_contributions"),
        round(avg("player_rating"), 2).alias("avg_player_rating"),
        round(avg("performance_score"), 2).alias("avg_performance_score"),
        round(avg("minutes_played"), 2).alias("avg_minutes_played")
    )
    .orderBy("avg_performance_score", ascending=False)
)

position_summary_df.write.mode("overwrite").format("delta").saveAsTable(gold_position_summary_table)

display(spark.table(gold_position_summary_table))


### 7.3 Gold table — Top players


In [0]:
# Player-level Gold table for top-player dashboard.
top_players_df = (
    fifa_silver_df
    .groupBy("player_id", "player_name", "team", "nationality", "position", "position_group")
    .agg(
        countDistinct("match_id").alias("matches_played"),
        spark_sum("minutes_played").alias("total_minutes_played"),
        spark_sum("goals").alias("total_goals"),
        spark_sum("assists").alias("total_assists"),
        spark_sum("goal_contribution").alias("total_goal_contributions"),
        round(avg("player_rating"), 2).alias("avg_player_rating"),
        round(avg("performance_score"), 2).alias("avg_performance_score"),
        max("market_value_eur").alias("market_value_eur")
    )
    .orderBy("total_goal_contributions", ascending=False)
)

top_players_df.write.mode("overwrite").format("delta").saveAsTable(gold_top_players_table)

display(spark.table(gold_top_players_table))


### 7.4 Gold table — Tournament stage summary


In [0]:
# Tournament-stage Gold table.
stage_summary_df = (
    fifa_silver_df
    .groupBy("tournament_stage")
    .agg(
        count("*").alias("player_match_records"),
        countDistinct("match_id").alias("total_matches"),
        spark_sum("goals").alias("total_goals"),
        spark_sum("assists").alias("total_assists"),
        round(avg("player_rating"), 2).alias("avg_player_rating"),
        round(avg("performance_score"), 2).alias("avg_performance_score"),
        max("top_speed_kmh").alias("max_top_speed_kmh"),
        round(avg("distance_covered_km"), 2).alias("avg_distance_covered_km")
    )
    .orderBy("total_goals", ascending=False)
)

stage_summary_df.write.mode("overwrite").format("delta").saveAsTable(gold_stage_summary_table)

display(spark.table(gold_stage_summary_table))


### 7.5 Gold table — Value and performance summary


In [0]:
# Player value and performance Gold table.
value_summary_df = (
    fifa_silver_df
    .groupBy("player_id", "player_name", "team", "position")
    .agg(
        max("market_value_eur").alias("market_value_eur"),
        round(avg("performance_score"), 2).alias("avg_performance_score"),
        round(avg("player_rating"), 2).alias("avg_player_rating"),
        spark_sum("goal_contribution").alias("total_goal_contributions"),
        spark_sum("minutes_played").alias("total_minutes_played")
    )
    .withColumn(
        "value_band",
        when(col("market_value_eur") >= 80000000, "Elite value")
        .when(col("market_value_eur") >= 40000000, "High value")
        .when(col("market_value_eur") >= 10000000, "Medium value")
        .otherwise("Lower value")
    )
    .orderBy("avg_performance_score", ascending=False)
)

value_summary_df.write.mode("overwrite").format("delta").saveAsTable(gold_value_summary_table)

display(spark.table(gold_value_summary_table))


## 8. Load data from an external API

This section uses the Open-Meteo API to load real weather data for Warsaw.

Steps:
1. Call the external API using Python `requests`
2. Save the raw JSON response to the volume
3. Convert the JSON response into a Spark DataFrame
4. Save API data as a Delta table
5. Create a daily weather summary Gold table


In [0]:
# API table names.
api_weather_bronze_table = f"{catalog}.{schema}.lab01_api_weather_bronze"
api_weather_daily_gold_table = f"{catalog}.{schema}.lab01_api_weather_daily_gold"

In [0]:
import requests
import json

# Open-Meteo API URL for Warsaw weather. No API key is required.
api_url = (
    "https://api.open-meteo.com/v1/forecast"
    "?latitude=52.23"
    "&longitude=21.01"
    "&hourly=temperature_2m,relative_humidity_2m,wind_speed_10m"
    "&timezone=Europe%2FWarsaw"
)

# Call external API.
response = requests.get(api_url, timeout=30)
response.raise_for_status()

weather_json = response.json()

# Save raw JSON response to the volume.
dbutils.fs.mkdirs(raw_api_path)

dbutils.fs.put(
    f"{raw_api_path}/open_meteo_warsaw_weather.json",
    json.dumps(weather_json, indent=2),
    overwrite=True
)

# Convert nested JSON hourly arrays to rows.
hourly = weather_json["hourly"]

weather_rows = list(
    zip(
        hourly["time"],
        hourly["temperature_2m"],
        hourly["relative_humidity_2m"],
        hourly["wind_speed_10m"]
    )
)

weather_df = spark.createDataFrame(
    weather_rows,
    ["weather_time", "temperature_2m", "relative_humidity_2m", "wind_speed_10m"]
)

# Add metadata and save as API Bronze Delta table.
weather_bronze_df = (
    weather_df
    .withColumn("weather_date", to_date(col("weather_time")))
    .withColumn("source_api", lit("Open-Meteo"))
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("load_date", current_date())
)

weather_bronze_df.write.mode("overwrite").format("delta").option("mergeSchema", "true").saveAsTable(api_weather_bronze_table)

display(spark.table(api_weather_bronze_table))


### 8.1 API Gold table — Daily weather summary


In [0]:
# Daily weather summary Gold table.
weather_daily_summary_df = (
    weather_bronze_df
    .groupBy("weather_date")
    .agg(
        round(avg("temperature_2m"), 2).alias("avg_temperature_2m"),
        min("temperature_2m").alias("min_temperature_2m"),
        max("temperature_2m").alias("max_temperature_2m"),
        round(avg("relative_humidity_2m"), 2).alias("avg_relative_humidity_2m"),
        round(avg("wind_speed_10m"), 2).alias("avg_wind_speed_10m")
    )
    .orderBy("weather_date")
)

weather_daily_summary_df.write.mode("overwrite").format("delta").saveAsTable(api_weather_daily_gold_table)

display(spark.table(api_weather_daily_gold_table))


## 9. Validate all created tables


In [0]:
%sql
SHOW TABLES IN dbr_dev.parvinbadalov;


In [0]:
%sql
SELECT *
FROM dbr_dev.parvinbadalov.lab01_fifa_team_summary_gold
ORDER BY total_goal_contributions DESC;


## 10. Dashboard-ready SQL queries

Use these queries in Databricks dashboard / AI BI dashboard.


In [0]:
%sql
-- Main dashboard query: Team performance
SELECT
  team,
  nationality,
  region,
  player_match_records,
  total_players,
  total_matches,
  total_goals,
  total_assists,
  total_goal_contributions,
  avg_player_rating,
  avg_performance_score,
  avg_pass_accuracy,
  avg_shots_accuracy_pct,
  highest_market_value_eur,
  avg_market_value_eur
FROM dbr_dev.parvinbadalov.lab01_fifa_team_summary_gold
ORDER BY total_goal_contributions DESC;


In [0]:
%sql
-- Dashboard query: Top players
SELECT
  player_name,
  team,
  nationality,
  position,
  matches_played,
  total_minutes_played,
  total_goals,
  total_assists,
  total_goal_contributions,
  avg_player_rating,
  avg_performance_score,
  market_value_eur
FROM dbr_dev.parvinbadalov.lab01_fifa_top_players_gold
ORDER BY total_goal_contributions DESC
LIMIT 25;


In [0]:
%sql
-- Dashboard query: Position summary
SELECT *
FROM dbr_dev.parvinbadalov.lab01_fifa_position_summary_gold
ORDER BY avg_performance_score DESC;


## 11. Suggested dashboard visuals

Create dashboard: `Parvin - Lab 1 FIFA Dashboard`

Recommended visuals:
1. Bar chart: `team` vs `total_goals`
2. Bar chart: `team` vs `total_assists`
3. Bar chart: `team` vs `total_goal_contributions`
4. Bar chart: `team` vs `avg_performance_score`
5. Bar chart: `player_name` vs `total_goal_contributions`
6. Table: full team summary
7. Filter: `team`
8. Filter: `region`
9. Filter: `nationality`


## 12. Delta Lake benefits

Delta Lake benefits used/explained in this lab:

- **ACID transactions**: reliable table writes.
- **Schema enforcement**: prevents unexpected schema issues from corrupting tables.
- **Time travel**: allows querying older versions of tables.
- **Scalable Delta tables**: efficient tables for Spark and SQL workloads.
- **Unity Catalog governance**: tables are saved under a governed catalog and schema.


## 13. Lab 1 completion checklist

This notebook covers the Lab 1 requirements:

- Git folder/notebook ready for commit
- Schema and volume setup
- CSV import
- Delta table creation
- Bronze, Silver, and Gold layers
- Multiple `select` examples
- Multiple `filter` examples
- Multiple `join` examples
- Multiple `groupBy` examples
- External API load
- Dashboard-ready Gold tables
- Delta Lake benefits explanation

Tables created:
- `dbr_dev.parvinbadalov.lab01_fifa_players_bronze`
- `dbr_dev.parvinbadalov.lab01_fifa_players_silver`
- `dbr_dev.parvinbadalov.lab01_fifa_team_summary_gold`
- `dbr_dev.parvinbadalov.lab01_fifa_position_summary_gold`
- `dbr_dev.parvinbadalov.lab01_fifa_top_players_gold`
- `dbr_dev.parvinbadalov.lab01_fifa_stage_summary_gold`
- `dbr_dev.parvinbadalov.lab01_fifa_value_summary_gold`
- `dbr_dev.parvinbadalov.lab01_api_weather_bronze`
- `dbr_dev.parvinbadalov.lab01_api_weather_daily_gold`
